# Quickstart 3 — period search

Three ways to find the period, and one honest way to say whether a peak means
anything: Lomb–Scargle on the RVs, the phase-distance-correlation (PDC)
periodogram on any channel, and the linearised Thiele–Innes frequency scan on
the astrometry. None of them computes a false-alarm probability for you; the
last cell shows the fifteen-line scramble null that does.

Units throughout: periods in **days**, frequencies in **cycles per day**, times in **MJD**, along-scan abscissae in **mas**, scan angles in **radians**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from orblet.simulate.bundles import load_simulated_inputs
from orblet import prepare_rv_for_orbit, resolve_epochs_mjd

bundle = load_simulated_inputs(seed=1)
truth = bundle.truth
astro = bundle.astro_data
prepared = prepare_rv_for_orbit(bundle.rv_data, time_scale="gaia_obmt")
EPOCH_REF = float(truth.t_ref_mjd)

t_rv, rv, rv_err = prepared["epochs_mjd"], prepared["rv"], prepared["rv_err"]
t_mjd = resolve_epochs_mjd(astro)
psi = np.asarray(astro["scan_angle"], dtype=float)
pf = np.asarray(astro["parallax_factor_al"], dtype=float)
d_obs = np.asarray(astro["centroid_pos"], dtype=float)
sigma = np.asarray(astro["centroid_pos_err"], dtype=float)
print(f"truth period {truth.P_days:.2f} d; RV epochs {t_rv.size}, astrometric epochs {t_mjd.size}, span {t_mjd.max() - t_mjd.min():.0f} d")

## Lomb–Scargle on the RVs

A sinusoid model. Fine for a first look; an eccentric orbit spreads power into
harmonics. `peak_fwhm_days` measures the width of the tallest peak — a
data-driven prior width for the next stage.

In [ ]:
from orblet.periodogram import compute_lomb_scargle_periodogram, peak_fwhm_days

# Plain arrays in, plain arrays out (the prepared RVs are unit-stripped km/s and MJD).
freq, power, best_f, best_P = compute_lomb_scargle_periodogram(rv, rv_err, t_rv)
freq = np.asarray(freq, dtype=float); power = np.asarray(power, dtype=float)
periods_ls = 1.0 / freq
P_peak, fwhm = peak_fwhm_days(periods_ls, power)
print(f"Lomb-Scargle best period {float(best_P):.2f} d (truth {truth.P_days:.2f}); peak FWHM {fwhm:.2f} d")
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogx(periods_ls, power); ax.axvline(truth.P_days, color="k", lw=0.8)
ax.set_xlabel("period (days)"); ax.set_ylabel("LS power")
plt.show()

## PDC: a periodogram for any channel

The phase-distance correlation needs only a **distance matrix** between
observations, so it works on RVs, on 1-D scan data, or on spectra. The kernels
are separate primitives: `scalar_distance_matrix` for scalars,
`scan_angle_distance_matrix` for the scan angle (ψ modulo π),
`astrometric_segment_distance_matrix` for along-scan abscissae. The scan-angle
kernel can be partialled out as a nuisance.

On the astrometry the search runs on the **residuals of the single-star
five-parameter solution**, never on the raw abscissae: position offset, proper
motion and parallax are a smooth secular signal that phases coherently at long
periods and drives any periodogram to the edge of the range.
`fit_astrometric_5param` is the closed-form solve. With a catalogue solution
in hand, subtract that instead — it is external to these data and cannot
absorb any of the orbit, whereas a fitted single-star model can take up a
little of it.

In [ ]:
from orblet import fit_astrometric_5param
from orblet.periodogram import (compute_pdc_periodogram, scalar_distance_matrix,
                                scan_angle_distance_matrix,
                                astrometric_segment_distance_matrix)

periods = np.geomspace(20.0, 2000.0, 600)
pdc_rv = compute_pdc_periodogram(t_rv, periods, scalar_distance_matrix(rv))
sol5 = fit_astrometric_5param(astro, epoch_ref_mjd=EPOCH_REF)
resid_al = np.asarray(sol5.residuals, dtype=float)
print(f"single-star fit: chi2/dof {sol5.chi2_per_dof:.2f}, residual rms {sol5.rms_mas:.3f} mas (orbit + noise)")
D_al = astrometric_segment_distance_matrix(resid_al, psi)
pdc_al = compute_pdc_periodogram(t_mjd, periods, D_al)
pdc_al_partial = compute_pdc_periodogram(t_mjd, periods, D_al,
                                         nuisance_dist=scan_angle_distance_matrix(psi),
                                         partial_mode="semi")
print(f"PDC best period: RV {pdc_rv['best_period_days']:.1f} d; along-scan {pdc_al['best_period_days']:.1f} d; "
      f"along-scan with scan angle partialled out {pdc_al_partial['best_period_days']:.1f} d (truth {truth.P_days:.1f})")
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogx(periods, pdc_rv["scores"], label="RV"); ax.semilogx(periods, pdc_al["scores"], label="along-scan")
ax.axvline(truth.P_days, color="k", lw=0.8); ax.set_xlabel("period (days)"); ax.set_ylabel("PDC score"); ax.legend()
plt.show()

## The Thiele–Innes frequency scan

`scan_ti_frequency` walks a frequency grid, and at each node a coarse
(e, τ) sub-grid, solving the nine linear amplitudes in closed form and ranking
nodes by `logL_marginal`. Read the ranking honestly: it is marginal over the
amplitudes but takes the **best** (e, τ) per frequency, so eccentric nodes are
favoured and a peak's `ecc` is not a measurement. Only the 1-year line and its
6-month harmonic are flagged; other aliases are yours to check.

In [ ]:
from orblet import scan_ti_frequency

SCAN = dict(f_min_per_day=1.0 / 2000.0, f_max_per_day=1.0 / 20.0, oversample=2.0,
            ecc_grid=[0.0, 0.3, 0.6], tau_grid=np.linspace(0.0, 1.0, 8, endpoint=False),
            epoch_ref_mjd=EPOCH_REF, top_k=5)
peaks = scan_ti_frequency(t_mjd, psi, pf, d_obs, sigma, **SCAN)
# A peak carries observables only: the shape node, the score, the amplitudes,
# and the fitted parallax with its formal sigma. Read the parallax next to a
# suspicious period -- a wrong period often comes with a parallax the fit cannot
# tell from zero. Nothing acts on it automatically; the seeded refiner and you do.
for k, pk in enumerate(peaks):
    print(f"peak {k}: P = {pk.P_days:8.2f} d, e-node = {pk.ecc:.1f}, logL_marginal = {pk.logL_marginal:8.1f}, "
          f"plx = {pk.plx_mas:6.3f} +/- {pk.plx_sigma_mas:.3f} mas, flagged = {pk.flagged}"
          f"{' ' + str(pk.flag_reasons) if pk.flagged else ''}")
print(f"truth: {truth.P_days:.2f} d, plx = {truth.parallax_mas:.3f} mas")

## Is the peak real? The scramble null

No function here computes a false-alarm probability. The honest substitute is
cheap: keep the epochs, scan angles and parallax factors, shuffle the abscissae
among them, and rescan. The best score of a scrambled data set is what noise
and the scan law alone can produce.

In [ ]:
rng = np.random.default_rng(0)
null_best = []
for _ in range(6):
    scrambled = rng.permutation(d_obs)
    pk_null = scan_ti_frequency(t_mjd, psi, pf, scrambled, sigma, **{**SCAN, "top_k": 1})
    null_best.append(pk_null[0].logL_marginal if pk_null else np.nan)
null_best = np.asarray(null_best)
print(f"best score, real data: {peaks[0].logL_marginal:.1f}")
print(f"best score, scrambled: median {np.nanmedian(null_best):.1f}, max {np.nanmax(null_best):.1f} over {null_best.size} scrambles")
print("margin of the real peak above the scrambled maximum (nats):", round(peaks[0].logL_marginal - np.nanmax(null_best), 1))

Next: `04_interpretation_mass.ipynb` — from amplitudes to a companion mass, and
what is assumed on the way.